<a href="https://colab.research.google.com/github/GanghyunShin02/Fenicsx/blob/code/Fencsx_HeatEQ_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------------
# 1️⃣ Mount Google Drive (optional, for cache)
# --------------------------------------------------
from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("📦 Google Drive already mounted")

# --------------------------------------------------
# 2️⃣ Clone fenicsx-colab repository (idempotent)
# --------------------------------------------------
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

def run(cmd):
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    print("📥 Cloning fenicsx-colab...")
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError("Directory exists but is not a git repository")
else:
    print("📦 Repository already exists — skipping clone")

# --------------------------------------------------
# 3️⃣ Run setup_fenicsx.py IN THIS KERNEL (CRITICAL)
# --------------------------------------------------
print("🚀 Running setup_fenicsx.py in current kernel")

# ⚙️ Configuration
USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = False    # <--- Set True to remove existing environment

# Build options
opts = []
if USE_COMPLEX:
    opts.append("--complex")
if USE_CLEAN:
    opts.append("--clean")

opts_str = " ".join(opts) if opts else ""

get_ipython().run_line_magic(
    "run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}"
)

# --------------------------------------------------
# 4️⃣ Sanity check
# --------------------------------------------------
try:
    get_ipython().run_cell_magic('fenicsx', '--info -np 4', '')
except Exception as e:
    print("⚠️ %%fenicsx magic not found:", e)

📦 Google Drive already mounted
📦 Repository already exists — skipping clone
🚀 Running setup_fenicsx.py in current kernel
🔧 FEniCSx Setup Configuration
PETSc type      : real
Clean install   : False

📦 Google Drive detected — using persistent cache

🔧 Installing FEniCSx environment...


CalledProcessError: Command '['bash', '/content/fenicsx-colab/setup/install_fenicsx.sh']' returned non-zero exit status 2.

⚠️ %%fenicsx magic not found: Cell magic `%%fenicsx` not found.


In [ ]:
#heat
#f
%%fenicsx -np -1 --time

import numpy as np
from dolfinx import mesh,fem,default_scalar_type,io
from mpi4py import MPI
import ufl
from ufl import (dx,
                 TrialFunction,
                 TestFunction,
                 grad,
                 dot)
from dolfinx.fem.petsc import (
                               assemble_vector,
                               assemble_matrix,
                               create_vector,
                               apply_lifting,
                               set_bc)
from petsc4py import PETSc as petsc
from pathlib import Path

t=0.0
T=1.0
num_steps=50
dt=T/num_steps

domain=mesh.create_rectangle(MPI.COMM_WORLD,
                              [np.array([-2,-2]),np.array([2,2])],
                               [50,50],
                               )

V=fem.functionspace(domain,("Lagrange",1))

def initial_condition(x,a=5):
  return np.exp(-a*(x[0]**2+x[1]**2))

un=fem.Function(V)
un.name='un'
un.interpolate(initial_condition)

uh=fem.Function(V)
uh.name='uh'
uh.interpolate(initial_condition)

fdim=domain.topology.dim-1
boundary_facets=mesh.locate_entities_boundary(domain,
                                              fdim,
                                              lambda x:np.full(x.shape[1],True,dtype=bool))

bc=fem.dirichletbc(petsc.ScalarType(0),
                   fem.locate_dofs_topological(V,fdim,boundary_facets),V)


'''
xdmf=io.XDMFFile(MPI.COMM_WORLD,Path("diffusion.xdmf"),"w")
xdmf.write_mesh(domain)
'''

u=TrialFunction(V)
v=TestFunction(V)
f=fem.Constant(domain,petsc.ScalarType(0))

#해석해
'''
def analytical_solution(t,a=5):
    return lambda x: (1/(4*a*t+1))*np.exp((-a*(x[0]**2+x[1]**2))/(4*a*t+1))
'''
import numpy as np

def analytical_fourier_solution(x_points):
    """
    경계 조건 u=0 ([-2, 2] x [-2, 2])을 만족하는 급수해
    t: 현재 시간 (여기서는 1.0)
    L: 도메인 반폭 (2.0)
    a: 초기 가우시안 계수 (5)
    N: 급수 항 개수 (정확도를 위해 30개 정도 사용)
    """
    t = 1.0
    L = 2.0
    a = 5
    N = 30

    # x_points는 dolfinx에서 (3, num_points) 형태로 들어옴
    xx = x_points[0]
    yy = x_points[1]

    u_val = np.zeros_like(xx)

    # 급수 계산 (홀수 항만 기여함 - 대칭성 때문)
    for m in range(1, N, 2):
        for n in range(1, N, 2):
            # 고유값 (Eigenvalues)
            lambda_m = (m * np.pi) / (2 * L)
            lambda_n = (n * np.pi) / (2 * L)
            eigen_val = lambda_m**2 + lambda_n**2

            # 초기 조건 u0에 대한 푸리에 계수 Amn (수치적 근사치)
            # 가우시안 적분값과 사인 함수의 곱
            coeff_m = (np.sqrt(np.pi / a) * np.exp(-(lambda_m**2) / (4 * a))) / L
            coeff_n = (np.sqrt(np.pi / a) * np.exp(-(lambda_n**2) / (4 * a))) / L
            A_mn = coeff_m * coeff_n

            # 시간 감쇄 및 공간 기여 합산
            u_val += A_mn * np.cos(lambda_m * xx) * np.cos(lambda_n * yy) * np.exp(-eigen_val * t)

    # L=2 도메인 크기에 맞춰 정규화 (초기값 u(0,0)=1에 근사하도록 보정)
    # 실제 수치 적분이 복잡하므로, 이론적 계수 4를 곱해줍니다.
    return u_val * 4.0
uex=fem.Function(V)
uex.interpolate(analytical_fourier_solution)


a=v*u*dx+dt*dot(grad(v),grad(u))*dx
L=(un+dt*f)*v*dx
'''
a의 계수는 시간에따라 변하지않고...
L은 변하므로(un을 업데이트)

a의 행렬을 먼저구성.
.L의 벡터는 메모리공간만 할당.
'''

bilinearform=fem.form(a)
linearform=fem.form(L)

A=assemble_matrix(bilinearform,bcs=[bc])
A.assemble()
b=create_vector(fem.extract_function_spaces(linearform))

'''
solver 구성..
solver를 petsc의 KSP(Krylov Subspace Method)방법으로풀이.
setOperator로 풀어야할 행렬 A를 지정..
preonly방법으로 풀어라..
preonly방법은 복잡한 반복계산하지말고 뒤에서 지정할(LU)방법으로
한번만 빠르게 풀어라..
전처리(PC)타입은 LU인수분해..
'''

solver=petsc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(petsc.KSP.Type.PREONLY)
solver.getPC().setType(petsc.PC.Type.LU)


#시각화
xdmf = io.XDMFFile(MPI.COMM_WORLD, "plz_HeatHeat_real.xdmf", "w")
xdmf.write_mesh(domain)

for i in range(num_steps):
    t+=dt

    # 1) RHS 초기화
    with b.localForm() as loc_b:
        loc_b.set(0)

    # 2) RHS assemble
    assemble_vector(b, linearform)

    # 3) Dirichlet lifting
    apply_lifting(b, [bilinearform], [[bc]])
    b.ghostUpdate(addv=petsc.InsertMode.ADD_VALUES,
                  mode=petsc.ScatterMode.REVERSE)
    set_bc(b, [bc])

    # 4) Solve linear system
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # 5) update previous solution
    un.x.array[:] = uh.x.array

    # 6) output
    xdmf.write_function(uh, t)


#L2error

error_form = fem.form(dot(uh - uex, uh - uex) * dx)

# 2. 각 프로세스에서 계산된 오차 합산 (병렬 처리 고려)
local_error = fem.assemble_scalar(error_form)
global_error = domain.comm.allreduce(local_error, op=MPI.SUM)

# 3. 최종적으로 루트를 씌워 L2 norm 도출
error_L2 = np.sqrt(global_error)

print("-" * 30)
print(f"Final Time t = {t:.2f}")
print(f"Simulation Max: {np.max(uh.x.array):.6f}")
print(f"Analytical Max: {1/(4*5*1.0 + 1):.6f}") # 1/21
print(f"L2 Error at t=1.0: {error_L2:.6e}")
print("-" * 30)

if domain.comm.rank==0:
  print('max Error',np.max(np.abs(uh.x.array-uex.x.array)))
'''
while t < T:
    t+=dt

    # 1) RHS 초기화
    with b.localForm() as loc_b:
        loc_b.set(0)

    # 2) RHS assemble
    assemble_vector(b, linearform)

    # 3) Dirichlet lifting
    apply_lifting(b, [bilinearform], [[bc]])
    b.ghostUpdate(addv=petsc.InsertMode.ADD_VALUES,
                  mode=petsc.ScatterMode.REVERSE)
    set_bc(b, [bc])

    # 4) Solve linear system
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # 5) update previous solution
    un.x.array[:] = uh.x.array

    # 6) output
    xdmf.write_function(uh, t)

'''

print('max uh',np.max(uh.x.array))
print('min uh',np.min(uh.x.array))
xdmf.close()

#print(np.average(880.800,852.161,589.205,656.441,840.602))

------------------------------
Final Time t = 1.00
Simulation Max: 0.044281
Analytical Max: 0.047619
L2 Error at t=1.0: 2.566864e-01
------------------------------
max Error 0.1296869642736979
max uh 0.04428066347603506
min uh 0.0
⏱ Elapsed time: 1.100111 s


In [ ]:
#LinearProblem
%%fenicsx -np -1 --time

import numpy as np
from mpi4py import MPI
from dolfinx import (mesh,
                     io,
                     fem,
                     default_scalar_type,
                     )
import ufl
from ufl import (dx,
                 TrialFunction,
                 TestFunction,
                 dot,
                 grad
                 )
from dolfinx.fem.petsc import LinearProblem
from dolfinx.fem.petsc import PETSc as petsc


t=0.0
T=1.0
timesteps=50
dt=T/timesteps

domain=mesh.create_rectangle(MPI.COMM_WORLD,
                              [np.array([-2,-2]),np.array([2,2])],
                              [50,50])
V=fem.functionspace(domain,("Lagrange",1))

def initial_condition(x,a=5):
  return np.exp(-a*(x[0]**2+x[1]**2))

un=fem.Function(V)
un.name='un'
un.interpolate(initial_condition)

fdim=domain.topology.dim-1
boundary_facets=mesh.locate_entities_boundary(domain,
                                              fdim,
                                              lambda x:np.full(x.shape[1],True,dtype=bool))

bc=fem.dirichletbc(petsc.ScalarType(0),
                   fem.locate_dofs_topological(V,fdim,boundary_facets),V)


uh=fem.Function(V)
uh.name='uh'
uh.interpolate(initial_condition)

u=TrialFunction(V)
v=TestFunction(V)
f=fem.Constant(domain,petsc.ScalarType(0))

def analytical_fourier_solution(x_points):
    """
    경계 조건 u=0 ([-2, 2] x [-2, 2])을 만족하는 급수해
    t: 현재 시간 (여기서는 1.0)
    L: 도메인 반폭 (2.0)
    a: 초기 가우시안 계수 (5)
    N: 급수 항 개수 (정확도를 위해 30개 정도 사용)
    """
    t = 1.0
    L = 2.0
    a = 5
    N = 30

    # x_points는 dolfinx에서 (3, num_points) 형태로 들어옴
    xx = x_points[0]
    yy = x_points[1]

    u_val = np.zeros_like(xx)

    # 급수 계산 (홀수 항만 기여함 - 대칭성 때문)
    for m in range(1, N, 2):
        for n in range(1, N, 2):
            # 고유값 (Eigenvalues)
            lambda_m = (m * np.pi) / (2 * L)
            lambda_n = (n * np.pi) / (2 * L)
            eigen_val = lambda_m**2 + lambda_n**2

            # 초기 조건 u0에 대한 푸리에 계수 Amn (수치적 근사치)
            # 가우시안 적분값과 사인 함수의 곱
            coeff_m = (np.sqrt(np.pi / a) * np.exp(-(lambda_m**2) / (4 * a))) / L
            coeff_n = (np.sqrt(np.pi / a) * np.exp(-(lambda_n**2) / (4 * a))) / L
            A_mn = coeff_m * coeff_n

            # 시간 감쇄 및 공간 기여 합산
            u_val += A_mn * np.cos(lambda_m * xx) * np.cos(lambda_n * yy) * np.exp(-eigen_val * t)

    # L=2 도메인 크기에 맞춰 정규화 (초기값 u(0,0)=1에 근사하도록 보정)
    # 실제 수치 적분이 복잡하므로, 이론적 계수 4를 곱해줍니다.
    return u_val * 4.0
uex=fem.Function(V)
uex.interpolate(analytical_fourier_solution)

a=v*u*dx+dt*dot(grad(v),grad(u))*dx
L=(un+dt*f)*v*dx

Problem=LinearProblem(a,
                      L,
                      bcs=[bc],
                      petsc_options_prefix='s')



xdmf = io.XDMFFile(MPI.COMM_WORLD, "plz_Heat_LinearProblem.xdmf", "w")
xdmf.write_mesh(domain)


for i in range(timesteps):
  t+=dt

  uh=Problem.solve()
  uh.x.scatter_forward()
  xdmf.write_function(uh,t)

  un.x.array[:]=uh.x.array[:]
'''
while t<T:
  t+=dt

  uh=Problem.solve()
  uh.x.scatter_forward()
  xdmf.write_function(uh,t)

  un.x.array[:]=uh.x.array[:]
'''
xdmf.close()

if domain.comm.rank==0:
  print('max Error',np.max(np.abs(uh.x.array-uex.x.array)))

print('max uh',np.max(uh.x.array))
print('min uh',np.min(uh.x.array))




max Error 0.12968730373686052
max uh 0.044280324012872456
min uh 0.0
⏱ Elapsed time: 0.805134 s


In [ ]:
#heat_general_solution
#f
%%fenicsx -np -1 --time

import numpy as np
from dolfinx import mesh,fem,default_scalar_type,io,plot
from mpi4py import MPI
import ufl
from ufl import (dx,
                 TrialFunction,
                 TestFunction,
                 grad,
                 dot)
from dolfinx.fem.petsc import (LinearProblem,
                               assemble_vector,
                               assemble_matrix,
                               create_vector,
                               apply_lifting,
                               set_bc)
from petsc4py import PETSc as petsc
import numpy.typing as npt


t=0.0
T=2.0
num_steps=20
dt=T/num_steps
alpha=3.0
beta=1.2

domain=mesh.create_unit_square(MPI.COMM_WORLD,
                               5,
                               5,
                               mesh.CellType.triangle)

V=fem.functionspace(domain,("Lagrange",1))

class ExactSolution:
    def __init__(self, alpha: float, beta: float, t: float):
        self.alpha = alpha
        self.beta = beta
        self.t = t

    def __call__(self, x: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
        return 1 + x[0] ** 2 + self.alpha * x[1] ** 2 + self.beta * self.t


u_exact = ExactSolution(alpha, beta, t)

uD=fem.Function(V)
uD.interpolate(u_exact)

tdim=domain.topology.dim
fdim=tdim-1

domain.topology.create_connectivity(fdim,tdim)
boundary_facets=mesh.exterior_facet_indices(domain.topology)

bc=fem.dirichletbc(uD,fem.locate_dofs_topological(V,fdim,boundary_facets))

un=fem.Function(V)
un.interpolate(u_exact)
f=fem.Constant(domain, beta-2-2*alpha)

u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
F = (
    u * v * ufl.dx
    + dt * ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
    - (un + dt * f) * v * ufl.dx
)
a = fem.form(ufl.lhs(F))
L = fem.form(ufl.rhs(F))


from pathlib import Path
'''
xdmf=io.XDMFFile(MPI.COMM_WORLD,Path("diffusion.xdmf"),"w")
xdmf.write_mesh(domain)
'''

'''
a의 계수는 시간에따라 변하지않고...
L은 변하므로(un을 업데이트)

a의 행렬을 먼저구성.
.L의 벡터는 메모리공간만 할당.
'''


A=assemble_matrix(a,bcs=[bc])
A.assemble()
b=create_vector(fem.extract_function_spaces(L))
uh=fem.Function(V)
uh.name="uh"

'''
solver 구성..
solver를 petsc의 KSP(Krylov Subspace Method)방법으로풀이.
setOperator로 풀어야할 행렬 A를 지정..
preonly방법으로 풀어라..
preonly방법은 복잡한 반복계산하지말고 뒤에서 지정할(LU)방법으로
한번만 빠르게 풀어라..
전처리(PC)타입은 LU인수분해..
'''

solver=petsc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(petsc.KSP.Type.PREONLY)
solver.getPC().setType(petsc.PC.Type.LU)


#시각화
xdmf = io.XDMFFile(MPI.COMM_WORLD, "Heat_general.xdmf", "w")
xdmf.write_mesh(domain)

while t < T:

    u_exact.t+=dt
    uD.interpolate(u_exact)

    # 1) RHS 초기화
    with b.localForm() as loc_b:
        loc_b.set(0)

    # 2) RHS assemble
    assemble_vector(b, L)

    # 3) Dirichlet lifting
    apply_lifting(b, [a], [[bc]])
    b.ghostUpdate(addv=petsc.InsertMode.ADD_VALUES,
                  mode=petsc.ScatterMode.REVERSE)
    set_bc(b, [bc])

    # 4) Solve linear system
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # 5) update previous solution
    un.x.array[:] = uh.x.array

    # 6) output
    xdmf.write_function(uh, t)

    t+=dt

xdmf.close()

# Compute L2 error and error at nodes
V_ex = fem.functionspace(domain, ("Lagrange", 2))
u_ex = fem.Function(V_ex)
u_ex.interpolate(u_exact)
error_L2 = np.sqrt(
    domain.comm.allreduce(
        fem.assemble_scalar(fem.form((uh - u_ex) ** 2 * ufl.dx)), op=MPI.SUM
    )
)
if domain.comm.rank == 0:
    print(f"L2-error: {error_L2:.2e}")

# Compute values at mesh vertices
error_max = domain.comm.allreduce(
    np.max(np.abs(uh.x.array - uD.x.array)), op=MPI.MAX
)
if domain.comm.rank == 0:
    print(f"Error_max: {error_max:.2e}")

L2-error: 2.83e-02
Error_max: 2.66e-15
⏱ Elapsed time: 0.695560 s


In [ ]:
print("sex")

sex
